# Game of Thrones Death Prediction — Exploratory Data Analysis

This notebook loads and inspects the raw JSON data for Game of Thrones episodes, scenes, and characters. It builds the processed DataFrames, examines class balance, and explores feature distributions. No model training occurs here — that work lives in `02_character_modeling.ipynb` and `03_scene_modeling.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Episodes

In this section I worked on preparing the dataframe for all of the episodes in the series.

In [ ]:
episodes = pd.read_json("../episodes.json")
episodes = pd.json_normalize(episodes["episodes"])
scenes = pd.DataFrame(episodes["scenes"])
scenes = scenes["scenes"]

In [ ]:
# Drop metadata columns that are not useful as features
episodes = episodes.drop(['episodeTitle',
                          'episodeLink',
                          'episodeAirDate',
                          'episodeDescription'], axis=1)

# Drop the list columns — scenes will be handled in their own dataframe
episodes = episodes.drop(['openingSequenceLocations', 'scenes'], axis=1)
episodes.head()

## Scenes

In this section I built the `all_scenes` dataframe, which contains one row per scene across all episodes of the series. I created several derived features:

- **Death in Scene** — whether any character died during the scene
- **sceneDuration** — scene length in seconds
- **totalCharacters** — number of characters in the scene
- **Season Number / Episode Number** — for later aggregation

In [ ]:
# Check total scene count to confirm the expected number of rows in all_scenes
num_scenes = 0
for episode in range(len(episodes)):
    num_scenes += len(scenes[episode])

print(f"Total scenes: {num_scenes}")

In [ ]:
# Build a dataframe of all scenes across the entire show
all_characters = []
all_scenes = pd.DataFrame(columns=scenes[0][0].keys())

for episode in range(len(episodes)):
    for scene in range(len(scenes[episode])):

        death_in_scene = False
        episode_number = int(episodes['episodeNum'][episode])
        season_number  = int(episodes['seasonNum'][episode])

        scene_characters = [character["name"] for character in scenes[episode][scene]["characters"]]
        all_characters.append(scene_characters)

        for character in scenes[episode][scene]["characters"]:
            if ('alive' in character.keys() and character['alive'] == False) and \
               ('mannerOfDeath' in character.keys() or 'killedBy' in character.keys()):
                death_in_scene = True

        scenes[episode][scene]['Death in Scene'] = death_in_scene
        scenes[episode][scene]['Season Number']  = season_number
        scenes[episode][scene]['Episode Number'] = episode_number

        del scenes[episode][scene]["characters"]

        scene_df = pd.DataFrame(scenes[episode][scene], index=[0])
        all_scenes = pd.concat([all_scenes, scene_df], ignore_index=True)

all_scenes["characters"] = all_characters

all_scenes['sceneStart'] = pd.to_timedelta(all_scenes['sceneStart']).dt.total_seconds()
all_scenes['sceneEnd']   = pd.to_timedelta(all_scenes['sceneEnd']).dt.total_seconds()
all_scenes['sceneDuration']   = all_scenes['sceneEnd'] - all_scenes['sceneStart']
all_scenes['totalCharacters'] = [len(c) for c in all_scenes['characters']]

all_scenes.head()

### Missing value check

Here I checked whether any of the numerical columns had missing data. Fortunately, none of them did.

In [ ]:
for col in ['sceneStart', 'sceneEnd', 'Season Number', 'Episode Number', 'totalCharacters']:
    n_null = all_scenes[col].isnull().sum()
    print(f"{col}: {n_null} nulls")

In [ ]:
# Distribution of scene locations
all_scenes['altLocation'].value_counts()

### One-hot encode the scenes DataFrame

I changed the non-numeric features (other than the characters list in each scene) to one-hot encoding so they will be usable by machine learning algorithms.

In [ ]:
# Drop the characters column when creating dummies because lists aren't hashable.
# Add the characters column back after encoding.
one_hot_scenes = all_scenes.drop(["characters"], axis=1)
one_hot_scenes = pd.get_dummies(one_hot_scenes)
one_hot_scenes["characters"] = all_characters
one_hot_scenes.head()

## Characters

I decided to get rid of several columns right off the bat: actor name, actor link, character link, characterImageThumb, characterImageFull, and nickname. While it might be possible to derive a metric for actor fame (whereby characters played by more famous actors might be less likely to die), it wasn't clear how I would obtain such a metric. The images and nickname also seemed irrelevant.

I added a number of derived binary features:

- **wasKilled** — the prediction target; whether the character was killed
- **hasKilledOthers** — whether the character killed anyone else
- **notHuman** — whether the character is one of the nine animals in the show (nearly all die)
- **wasServed**, **hasServed**, **hasSiblings**, **isYoungVersion**, **hasChildren**, etc.

In [ ]:
characters = pd.read_json("../characters.json")
characters = pd.json_normalize(characters["characters"])

In [ ]:
# Drop metadata/image columns that cannot be used as features
characters = characters.drop(['characterLink',
                              'actorName',
                              'actorLink',
                              'characterImageThumb',
                              'characterImageFull',
                              'nickname'], axis=1)
characters.head()

In [ ]:
# Add derived binary features
characters['wasKilled']   = characters['killedBy'].notnull()
characters['hasKilledOthers'] = characters['killed'].notnull()

# Determine if the character is one of the nine animals in the show
animals = ['Grey Wind', 'Lady', 'Nymeria', 'Summer', 'Shaggydog',
           'Ghost', 'Drogon', 'Rhaegal', 'Viserion']
is_animal = []
for row in characters['siblings']:
    if type(row) == list:
        is_animal.append(not set(animals).isdisjoint(row))
    else:
        is_animal.append(False)

characters['notHuman']            = is_animal
characters['wasServed']           = characters['servedBy'].notnull()
characters['hasSiblings']         = characters['siblings'].notnull()
characters['isYoungVersion']      = ['Young' in name for name in characters['characterName']]
characters['hasChildren']         = characters['parentOf'].notnull()
characters['childOfNamedCharacter'] = characters['parents'].notnull()
characters['isMarried']           = characters['marriedEngaged'].notnull()
characters['hasServed']           = characters['serves'].notnull()
characters['wasGuarded']          = characters['guardedBy'].notnull()
characters['wasGuardian']         = characters['guardianOf'].notnull()
characters['isRoyal']             = characters['royal'].notnull()
characters['isKingsguard']        = characters['kingsguard'].notnull()

characters.head()

### Screen time

Total screen time throughout the series may be an important feature for predicting whether a character is killed, so I calculated it from the scenes data and added it to the characters DataFrame.

In [ ]:
screen_times = []
for character in characters['characterName']:
    screen_time = np.sum(
        one_hot_scenes[[character in c for c in all_scenes['characters']]]['sceneDuration']
    )
    screen_times.append(screen_time)

characters['screenTime'] = screen_times
print(f"Max screen time: {np.max(characters['screenTime']):.0f} seconds")
characters[['characterName', 'screenTime']].sort_values('screenTime', ascending=False).head(10)

### Total scene screen time

I added one more derived feature to `one_hot_scenes`: the sum of the career screen times for all characters in each scene. The idea is that scenes containing characters with larger total screen times are less likely to contain a death — those characters are clearly around for a while.

In [ ]:
scene_screen_times = []
for scene_characters in all_scenes['characters']:
    total = 0
    for character in scene_characters:
        total += np.sum(characters[characters['characterName'] == character]['screenTime'])
    scene_screen_times.append(total)

one_hot_scenes['totalScreenTime'] = scene_screen_times
one_hot_scenes.head()

### Column investigation

In this section I investigated the characters dataframe columns to understand if they were necessary and whether it made sense to create any derived features. I knew that I eventually wanted to one-hot encode the DataFrame, so I couldn't have any lists remaining. My investigation was mostly used for seeing what type of data I was dealing with — I did not make statistical decisions based on this until after splitting into training and test sets.

In [ ]:
# The abducted, abductedBy, and sibling columns only apply to one character each
print("abductedBy:", characters['abductedBy'].value_counts().to_dict())
print("abducted:",   characters['abducted'].value_counts().to_dict())
print("sibling:",    characters['sibling'].value_counts().to_dict())

In [ ]:
# Confirm that there are exactly nine non-human characters
characters['notHuman'].value_counts()

In [ ]:
# Drop list-valued columns and columns that only apply to one or two characters
characters = characters.drop(['parents', 'royal', 'siblings', 'killedBy',
                              'killed', 'servedBy', 'parentOf',
                              'marriedEngaged', 'kingsguard', 'serves',
                              'guardedBy', 'actors', 'guardianOf',
                              'allies', 'abductedBy', 'abducted',
                              'sibling'], axis=1)
characters.head()

### Resolve multi-house characters

I was getting an error that there was still a list in the data when trying to one-hot encode the characters DataFrame. It turns out several characters (Jon Snow, Catelyn Stark, Cersei Lannister, Lysa Arryn, and Walda Bolton) belonged to multiple houses. I made a choice for each: they are placed under whichever house they most strongly align with.

In [ ]:
# Show all characters with list-valued houseName
characters[[type(house) == list for house in characters['houseName']]][['characterName', 'houseName']]

In [ ]:
# Assign each character to a single house
characters.loc[characters['characterName'] == 'Jon Snow',         ['houseName']] = 'Targaryen'
characters.loc[characters['characterName'] == 'Catelyn Stark',    ['houseName']] = 'Stark'
characters.loc[characters['characterName'] == 'Cersei Lannister', ['houseName']] = 'Lannister'
characters.loc[characters['characterName'] == 'Lysa Arryn',       ['houseName']] = 'Arryn'
characters.loc[characters['characterName'] == 'Walda Bolton',     ['houseName']] = 'Bolton'

# Verify no list-valued houses remain
remaining = characters[[type(house) == list for house in characters['houseName']]]
print(f"Characters still with list houseName: {len(remaining)}")

### One-hot encode the characters DataFrame

In [ ]:
one_hot_characters = pd.get_dummies(characters)
print(f"Shape: {one_hot_characters.shape}")
one_hot_characters.head()

## Class Balance Analysis

### Characters — `wasKilled`

The classes appear well balanced: approximately half of all characters in the show are killed.

In [ ]:
one_hot_characters['wasKilled'].value_counts().plot.bar()
plt.ylabel('Number of Characters')
plt.xlabel('Character Died (T/F)')
plt.title('Class Balance — wasKilled (characters)')
plt.tight_layout()
plt.show()

character_baseline = np.max(one_hot_characters['wasKilled'].value_counts()) / len(one_hot_characters)
print(f"Majority-class baseline accuracy: {character_baseline:.4f}")

### Scenes — `Death in Scene`

The classes are clearly very imbalanced — most scenes do not contain a character death. This will make prediction more difficult and will magnify false negatives (scenes where a death occurred that the model misses).

In [ ]:
# Fix the Death in Scene column after one-hot encoding split it in two
one_hot_scenes['deathInScene'] = one_hot_scenes['Death in Scene_True']
one_hot_scenes = one_hot_scenes.copy()
one_hot_scenes = one_hot_scenes.drop(['Death in Scene_False', 'Death in Scene_True'], axis=1)

one_hot_scenes['deathInScene'].value_counts().plot.bar()
plt.ylabel('Number of Scenes')
plt.xlabel('Character Death in Scene (T/F)')
plt.title('Class Balance — deathInScene (scenes)')
plt.tight_layout()
plt.show()

scenes_baseline = np.max(one_hot_scenes['deathInScene'].value_counts()) / len(one_hot_scenes)
print(f"Majority-class baseline accuracy: {scenes_baseline:.4f}")

## Deaths per Episode

I added columns to the episodes DataFrame indicating how many deaths occur in each episode and whether any deaths occur at all. I needed to wait until the `Death in Scene` column was complete in `all_scenes` before I could compute these.

In [ ]:
deaths_per_episode = []

for season in episodes['seasonNum'].unique():
    for episode in range(1, np.max(episodes[episodes['seasonNum'] == season]['episodeNum'].unique()) + 1):
        deaths_per_episode.append(
            np.sum(
                all_scenes[
                    (all_scenes['Season Number'] == season) &
                    (all_scenes['Episode Number'] == episode)
                ]['Death in Scene']
            )
        )

episodes['deathsPerEpisode']      = deaths_per_episode
episodes['episodeContainsDeath']  = episodes['deathsPerEpisode'] > 0
episodes.head(20)

### Deaths per episode over the series

In [ ]:
plt.scatter(episodes.index.values, episodes['deathsPerEpisode'])
plt.xlabel('Episode Number')
plt.ylabel('Number of Deaths')
plt.title('Deaths per Episode in Game of Thrones')
plt.tight_layout()
plt.show()

print(f"Average deaths per episode: {np.mean(episodes['deathsPerEpisode']):.2f}")

## Feature Distribution — Character Screen Time

I thought about binning character screen time but decided against it, since the distribution is so heavily right-skewed that standardisation makes more sense for model training.

In [ ]:
bins = pd.cut(one_hot_characters['screenTime'], 8)
bins.value_counts().sort_index()

In [ ]:
one_hot_characters['screenTime'].hist(bins=30)
plt.xlabel('Total Screen Time (seconds)')
plt.ylabel('Number of Characters')
plt.title('Distribution of Character Screen Time')
plt.tight_layout()
plt.show()

## Summary

Key findings from this EDA:

- **389 characters**, **4 165 scenes**, and **73 episodes** in the dataset.
- The character survival target (`wasKilled`) is nearly balanced (~50/50), giving a majority-class baseline around 0.50.
- The scene death target (`deathInScene`) is heavily imbalanced — deaths occur in fewer than 5% of scenes — giving a baseline above 0.95.
- Screen time is heavily right-skewed: most characters appear very briefly, while a handful of leads accumulate thousands of seconds.
- Deaths are not uniformly distributed across the series; certain episodes have notably higher kill counts.

The processed parquet files in `reports/` are used for all subsequent modeling.